# Big Muff Finite Impulse Response Model Training
This notebook demonstrates how to use lyrebird to train a feed forward neural network to replicate the Big Muff audio effect and apply the model to transform audio.

## Step 1: Import Libraries
Import the `lyrebird` module which contains the types and functions for training an audio effect simulation.


In [ ]:
import lyrebird_audio as lyrebird

## Step 2: Configure Model Parameters
Set the `buffer_length` - the number of input samples the model uses to predict each output sample.


In [2]:
# Configuration parameters
buffer_length = 512  # Number of input samples to use as context for the model

print(f"Configuration:")
print(f"  Buffer Length: {buffer_length} samples")


Configuration:
  Buffer Length: 512 samples


## Step 3: Create Training Dataset
Load paired audio files: a clean input and the desired output (with the Big Muff effect applied).

**Training files:**
- Input: `"../data/Big Muff piano clean.wav"` - clean piano audio
- Output: `"../data/Big Muff piano distorted.wav"` - same audio processed through Big Muff effect
- Both files must:
  - Have the same sample rate
  - Have the same length (number of samples)
  - Be time-aligned (no delay between input and output)

In [ ]:
training_dataset = lyrebird.FiniteImpulseResponseDataSet(
    input_wav_path="../data/Big Muff piano clean.wav",
    output_wav_path="../data/Big Muff piano distorted.wav",
    buffer_length=buffer_length
)

## Step 4: Inspect Dataset
Verify the dataset loaded correctly and check the data shapes.


In [4]:
# Log dataset information
print(f"Dataset length: {len(training_dataset)}")
print(f"Sample rate: {training_dataset.get_sample_rate()} Hz")
print(f"Input channels: {training_dataset.get_input_channels()}")
print(f"Output channels: {training_dataset.get_output_channels()}")
print(f"Total samples: {training_dataset.get_total_samples()}")

# Get a sample to inspect shapes
input_buffer, output_sample = training_dataset[0]
print(f"\nTraining data shapes:")
print(f"  Input buffer shape: {input_buffer.shape}")
print(f"  Output sample shape: {output_sample.shape}")


Dataset length: 3268459
Sample rate: 44100 Hz
Input channels: 1
Output channels: 1
Total samples: 3268971

Training data shapes:
  Input buffer shape: torch.Size([1, 512])
  Output sample shape: torch.Size([1])


## Step 5: Create DataLoader
Set up batched training with PyTorch's DataLoader.

**For different effects, you may want to adjust:**
- `batch_size`: 32 is good for most cases
  - Increase (64, 128) for faster training with more GPU memory
  - Decrease (16, 8) if running out of memory
- `num_workers`: Number of CPU threads for data loading
  - Adjust based on your CPU cores (typically 2-8)
- Keep `shuffle=True` for better training since the model does not have internal state
- Keep `pin_memory=True` when using GPU


In [5]:
import torch
from torch.utils.data import DataLoader

# Create a DataLoader
train_loader = DataLoader(
    training_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

print(f"DataLoader created with {len(train_loader)} batches")
print(f"Batch size: {train_loader.batch_size}")

# Get a sample batch to inspect shapes
for input_batch, output_batch in train_loader:
    print(f"\nBatch shapes:")
    print(f"  Input batch shape: {input_batch.shape}")
    print(f"  Output batch shape: {output_batch.shape}")
    break


DataLoader created with 102140 batches
Batch size: 32

Batch shapes:
  Input batch shape: torch.Size([32, 1, 512])
  Output batch shape: torch.Size([32, 1])


## Step 6: Create and Configure Model
Initialize the neural network architecture.

The model automatically adapts to mono/stereo based on your audio files.


In [6]:
# Get device (GPU if available, otherwise CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Create the model
num_channels = training_dataset.get_input_channels()
input_size = buffer_length * num_channels

model = lyrebird.FiniteImpulseResponseModel(
    input_size=input_size,  # buffer_length * channels
    hidden_size=128,
    num_layers=3,
    output_size=training_dataset.get_output_channels()
)

# Move model to device
model = model.to(device)

print(f"\nModel configuration:")
print(f"  Input size: {input_size} ({buffer_length} × {num_channels} channels)")
print(f"  Output size: {training_dataset.get_output_channels()}")
print(f"\nModel architecture:")
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")


Using device: cuda

Model configuration:
  Input size: 512 (512 × 1 channels)
  Output size: 1

Model architecture:
FiniteImpulseResponseModel(
  (network): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=512, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=128, bias=True)
    (4): ReLU()
    (5): Linear(in_features=128, out_features=128, bias=True)
    (6): ReLU()
    (7): Linear(in_features=128, out_features=1, bias=True)
  )
)

Total parameters: 98,817


## Step 7: Setup Training
Configure the loss function and optimizer.


In [7]:
# Define loss function and optimizer
loss_fn = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print("Training setup complete!")
print(f"Loss function: {loss_fn}")
print(f"Optimizer: {optimizer.__class__.__name__}")
print(f"Learning rate: {optimizer.param_groups[0]['lr']}")


Training setup complete!
Loss function: MSELoss()
Optimizer: Adam
Learning rate: 0.001


## Step 8: Train the Model
Train the model over multiple epochs.


In [8]:
# Train the model
epochs = 5
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    lyrebird.train_epoch(train_loader, model, loss_fn, optimizer, device)
    lyrebird.evaluate(train_loader, model, loss_fn, device)
print("Done!")


Epoch 1
-------------------------------
loss: 0.040039  [      0/3268459]
loss: 0.001137  [ 160000/3268459]
loss: 0.000796  [ 320000/3268459]
loss: 0.000199  [ 480000/3268459]
loss: 0.000188  [ 640000/3268459]
loss: 0.000135  [ 800000/3268459]
loss: 0.000238  [ 960000/3268459]
loss: 0.000442  [1120000/3268459]
loss: 0.000357  [1280000/3268459]
loss: 0.000425  [1440000/3268459]
loss: 0.000216  [1600000/3268459]
loss: 0.000258  [1760000/3268459]
loss: 0.000323  [1920000/3268459]
loss: 0.000282  [2080000/3268459]
loss: 0.000159  [2240000/3268459]
loss: 0.000217  [2400000/3268459]
loss: 0.000292  [2560000/3268459]
loss: 0.000142  [2720000/3268459]
loss: 0.000103  [2880000/3268459]
loss: 0.000112  [3040000/3268459]
loss: 0.000275  [3200000/3268459]
Test Error: 
 Avg loss: 0.000204 

Epoch 2
-------------------------------
loss: 0.000465  [      0/3268459]
loss: 0.000216  [ 160000/3268459]
loss: 0.000132  [ 320000/3268459]
loss: 0.000219  [ 480000/3268459]
loss: 0.000099  [ 640000/3268459]
l

## Step 9: Save the Trained Model
Save the model weights and configuration for later use.

The saved file contains:
- Model weights (learned parameters)
- Optimizer state (for resuming training)
- Configuration (buffer_length, architecture details)

This allows you to load and use the model later without retraining.


In [ ]:
# Save the trained model
model_path = "../models/big_muff_fir_model.pth"

torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'buffer_length': buffer_length,
    'input_size': input_size,
    'hidden_size': 128,
    'num_layers': 3,
    'output_size': training_dataset.get_output_channels(),
}, model_path)

print(f"Model saved to: {model_path}")
print(f"Saved configuration:")
print(f"  Buffer length: {buffer_length}")
print(f"  Input size: {input_size}")
print(f"  Output size: {training_dataset.get_output_channels()}")

## Load Model (Optional)
To load the saved model later, use:
```python
checkpoint = torch.load("../models/big_muff_fir_model.pth")

# Recreate the model
loaded_model = lyrebird.FiniteImpulseResponseModel(
    input_size=checkpoint['input_size'],
    hidden_size=checkpoint['hidden_size'],
    num_layers=checkpoint['num_layers'],
    output_size=checkpoint['output_size']
)

# Load the trained weights
loaded_model.load_state_dict(checkpoint['model_state_dict'])
loaded_model = loaded_model.to(device)
loaded_model.eval()
```

## Step 10: Apply Model to Training Input
Test the trained model by applying it to the original training input file.


In [ ]:
# Apply the trained model to transform an audio file
output_file = "../outputs/Big Muff piano_transformed.wav"

print(f"Applying trained model to transform {training_dataset.input_wav_path}...")
lyrebird.transform(
    model=model,
    input_wav_path="../data/Big Muff piano clean.wav",
    output_wav_path=output_file,
    buffer_length=buffer_length,
    device=device
)

print(f"\nTransformed audio saved to: {output_file}")

## Step 11: Compare Results
Listen to and compare the input, target output, and model's output.

**What to listen for:**
1. **Input**: The clean/dry signal
2. **Target Output**: The desired Big Muff effect (what you trained on)
3. **Model Simulation**: What your model learned to produce


In [ ]:
from IPython.display import Audio, display
import IPython.display as ipd

# Display audio players for comparison
print("=" * 60)
print("AUDIO COMPARISON")
print("=" * 60)

print("\n1. INPUT (Clean Piano):")
display(Audio("../data/Big Muff piano clean.wav"))

print("\n2. TARGET OUTPUT (Piano with Big Muff Effect):")
display(Audio("../data/Big Muff piano distorted.wav"))

print("\n3. MODEL SIMULATION (Transformed by Model):")
display(Audio(output_file))

## Step 12: Test Generalization
Apply the trained model to data it has not been trained on to see how well it generalizes.

We'll test on the organ audio file, which is different from the piano used for training.


In [ ]:
# Apply the trained model to the organ audio file
organ_output_file = "../outputs/Big Muff organ_transformed.wav"

print(f"Applying trained model to transform Big Muff organ clean.wav...")
lyrebird.transform(
    model=model,
    input_wav_path="../data/Big Muff organ clean.wav",
    output_wav_path=organ_output_file,
    buffer_length=buffer_length,
    device=device
)

print(f"\nTransformed organ audio saved to: {organ_output_file}")

## Step 13: Compare Generalization Results
Listen to how the model performs on the test audio.

**What to evaluate:**
1. **Input**: Clean test audio (different from training)
2. **Target Output**: The actual Big Muff effect applied to this test audio
3. **Model Simulation**: Your model's attempt at the effect


In [ ]:
# Display audio players for organ comparison
print("=" * 60)
print("ORGAN AUDIO COMPARISON")
print("=" * 60)

print("\n1. INPUT (Clean Organ):")
display(Audio("../data/Big Muff organ clean.wav"))

print("\n2. TARGET OUTPUT (Organ with Big Muff Effect):")
display(Audio("../data/Big Muff organ distorted.wav"))

print("\n3. MODEL SIMULATION (Transformed by Model):")
display(Audio(organ_output_file))